# Bee1004 — annotated demo (runs on a saved volume)

This is the crop-free version of the demo. It loads an **already-extracted volume**
(`bee1004_demo/crop_volume.tif`, produced by the original demo notebook) and analyses it
directly — no reading from the 38 GB file, no crop box, no Google Drive permission needed.

Every code cell is annotated line-by-line so you can read exactly what each step does.

**What it produces:**
1. Bouton + axon detection with real counts (the tools that actually fit PDF+ structure)
2. A **brighter** rotating 3D GIF
3. A numbers table for the slide

Point `VOLUME` at any 3D TIFF that fits in memory to reuse this on other data.

## 0. Setup and load the volume

In [ ]:
# --- standard scientific-Python imports -----------------------------------------
import sys, time, warnings
from pathlib import Path

import numpy as np              # arrays
import pandas as pd             # the measurements table
import matplotlib.pyplot as plt # figures
import tifffile                 # reads the volume

warnings.filterwarnings("ignore")           # silence library deprecation chatter

# --- make the `hive` package importable regardless of where Jupyter was launched --
# Walk up from the current directory until we find the project (the folder holding
# the `hive/` package), then put it on the import path.
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "hive").is_dir():
        sys.path.insert(0, str(_c)); PROJECT = _c; break
else:
    raise SystemExit("Run this notebook from inside the HIVE_Final project folder.")

plt.rcParams.update({"figure.dpi": 120, "image.cmap": "magma"})   # nicer default plots

# --- the inputs -----------------------------------------------------------------
VOLUME   = PROJECT / "bee1004_demo" / "crop_volume.tif"   # <- point at any 3D TIFF
OUT      = PROJECT / "bee1004_demo"; OUT.mkdir(exist_ok=True)
PIXEL_XY = 0.161636973345304     # microns per pixel in x/y (from the OME metadata)
PIXEL_Z  = 0.6                   # microns per z-step

# --- load the volume into memory ------------------------------------------------
# tifffile.imread returns a (Z, Y, X) array. This volume is small enough to hold
# whole; that is the point of having cropped/extracted it beforehand.
volume = tifffile.imread(VOLUME).astype(np.float32)
print(f"loaded {VOLUME.name}: shape {volume.shape}  "
      f"({volume.nbytes/1e6:.0f} MB), intensity range [{volume.min():.0f}, {volume.max():.0f}]")

## 1. Background-correct, then max-project

Two steps, both necessary for thin PDF+ structure:

- **Background subtraction** (rolling-ball / top-hat) removes the uneven neuropil glow, so
  faint boutons and axons stand out. It's applied per plane.
- **Max projection** collapses the z-planes into one image by taking the brightest voxel at
  each pixel. An axon travels *through* z, so any single plane clips it — projecting makes
  the whole tract continuous.

In [ ]:
from hive.baseline import apply_baseline_subtraction   # the background remover

# Correct each z-plane independently, then stack the corrected planes back into a volume.
# apply_baseline_subtraction(method="tophat") estimates the smooth background with a
# morphological opening (radius = tophat_radius) and subtracts it. return_baseline=False
# means "don't hand back the background image, just the corrected plane".
TOPHAT_RADIUS = 25    # px; must be LARGER than your largest real object, or it eats signal

corrected = np.stack([
    apply_baseline_subtraction(volume[z], method="tophat",
                               tophat_radius=TOPHAT_RADIUS, return_baseline=False)[0]
    for z in range(volume.shape[0])          # loop over every z-plane
])

# Max-intensity projection: for each (y, x) keep the brightest value across all z.
projection = corrected.max(axis=0)
print(f"corrected + projected to a single {projection.shape} image")

## 2. Detect boutons and axons — the money cell

PDF+ arbors have two structure types that need two different detectors (plain Otsu +
`min_size` handles neither — that's why the original run reported 0):

- **boutons** (bright puncta): threshold the brightest ~1% of pixels
- **axons** (thin, dim tracts): a **Sato ridge filter** that keys on tube-like *shape*,
  not brightness

In [ ]:
from hive.neurites import detect_neurites      # boutons + axons in one call
from hive.cleanup import normalize_for_display  # stretches an image to [0,1] for display

# detect_neurites runs both detectors on the projection:
#   bright_percentile=99  -> the top 1% of pixels are candidate boutons
#   bouton_min_size=4     -> drop specks smaller than 4 px (boutons are tiny, keep it low)
#   axon_scales           -> tube half-widths (px) the ridge filter looks for; 1-3 = fine axons
#   axon_min_length=50    -> drop ridge fragments shorter than 50 px (real axons are long)
neu = detect_neurites(projection,
                      bright_percentile=99.0, bouton_min_size=4,
                      axon_scales=(1, 1.5, 2, 2.5, 3), axon_min_length=50)

# axon_length_px is the skeleton (1-px centreline) length; multiply by pixel size for microns.
axon_um = neu.axon_length_px * PIXEL_XY
print(f"boutons: {neu.n_boutons}")
print(f"axon skeleton: {neu.axon_length_px} px = {axon_um:.0f} um of PDF+ axon")

# --- build the three-panel figure ----------------------------------------------
disp = normalize_for_display(projection)     # background image, stretched for viewing

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(disp); ax[0].set_title("PDF+ signal (max projection)")

# boutons in cyan on a dimmed copy of the signal
o1 = np.dstack([disp * 0.7] * 3)             # grey background at 70% brightness
o1[neu.bouton_mask] = [0, 1, 1]              # paint detected bouton pixels cyan
ax[1].imshow(o1); ax[1].set_title(f"boutons: {neu.n_boutons}")

# axons in red, with the 1-px skeleton in yellow on top
o2 = np.dstack([disp * 0.6] * 3)
o2[neu.axon_mask] = o2[neu.axon_mask] * 0.3 + np.array([0.7, 0, 0])   # red tint on axon pixels
o2[neu.skeleton] = [1, 0.9, 0]               # yellow centreline
ax[2].imshow(o2); ax[2].set_title(f"axons: {axon_um:.0f} um skeleton")

for a in ax: a.axis("off")
plt.tight_layout()
plt.savefig(OUT / "boutons_axons.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"saved {OUT/'boutons_axons.png'}")

**Tuning if it looks off:** boutons too sparse → lower `bright_percentile` (98, 97);
axon trace broken up → lower `axon_min_length` or add a larger `axon_scales` value for
thicker tracts. Report **axon length**, not filament *count* — thin ridges fragment on
noise, so the count is unstable but the total length is robust.

## 3. The brighter rotating 3D GIF

A true 3D reconstruction spun through 360°. The brightness comes from two `RenderConfig`
knobs added for exactly this:

- **`gamma`** — below 1 brightens the dim midtones (the neurites). `0.6` is the sweet spot;
  lower blows out the background.
- **`contrast_percentile`** — normalise to a percentile instead of the max, so a few hot
  voxels don't darken everything. Left at 100 here; drop to ~99.5 for even more lift.

In [ ]:
from hive.pipeline import analyze_zstack
from hive.config import (ArborConfig, HiveConfig, PreprocessConfig,
                         RenderConfig, SegmentationConfig)

# analyze_zstack reads a volume from disk, so write the (small) corrected volume out once.
# imagej=True + axes="ZYX" tags it so viewers open it with the right orientation.
vol_path = OUT / "corrected_volume.tif"
tifffile.imwrite(vol_path, corrected.astype(np.float32), imagej=True, metadata={"axes": "ZYX"})

cfg = HiveConfig(
    input_path=str(vol_path), output_folder=str(OUT / "3d"), mode="3d", channel="gray",
    voxel_size_xy=PIXEL_XY, voxel_size_z=PIXEL_Z, voxel_unit="um",
    # the volume is ALREADY background-corrected above, so DON'T subtract again --
    # a second subtraction crushes the signal and darkens the GIF. "none" keeps it as-is.
    preprocess=PreprocessConfig(baseline_method="none", gaussian_sigma=0, use_clahe=False),
    segmentation=SegmentationConfig(min_size=100),
    arbor=ArborConfig(enabled=False),
    render=RenderConfig(
        mode="mip",            # maximum-intensity projection per frame
        n_frames=36,           # 36 frames = 10 degrees apart = smooth rotation
        fps=15,                # playback speed
        colormap="magma",      # any matplotlib colormap
        downsample=2,          # shrink 2x before rendering (faster, smaller GIF)
        gamma=0.6,             # <-- BRIGHTNESS: <1 brightens the dim neurites
        # contrast_percentile=99.5,   # <-- uncomment for an extra brightness boost
    ),
)

t0 = time.perf_counter()
res3d = analyze_zstack(cfg, render_gif=True)     # segments + renders the GIF
print(f"3D done in {time.perf_counter()-t0:.0f}s — {res3d.object_count} objects")

# show the GIF inline
from IPython.display import Image as IPyImage, display as ipy_display
gif = res3d.outputs.get("gif")
if gif and Path(gif).exists():
    print(f"GIF: {gif}")
    ipy_display(IPyImage(filename=str(gif)))

## 4. Numbers for the slide

In [ ]:
# Pull the concrete figures together.
side_um = projection.shape[0] * PIXEL_XY
print(f"Region analysed: {side_um:.0f} x {side_um:.0f} um of real bee brain\n")
print(f"  PDF+ boutons        : {neu.n_boutons}")
print(f"  PDF+ axon length    : {axon_um:.0f} um")
print(f"  3D objects (volume) : {res3d.object_count}")

# save a one-row summary CSV
summary = pd.DataFrame([{
    "region_um": round(side_um, 1),
    "boutons": neu.n_boutons,
    "axon_length_um": round(axon_um, 1),
    "objects_3d": res3d.object_count,
}])
summary.to_csv(OUT / "demo_summary.csv", index=False)
display(summary)
print(f"""
Files in {OUT.name}/:
  boutons_axons.png       – the detection figure
  3d/HIVE_3D_rotation.gif  – the brightened rotating reconstruction
  3d/HIVE_3D_viewer.html   – interactive 3D (open in a browser, spin it live)
  demo_summary.csv         – the numbers above
""")

---
### For the talk

Real confocal data — honeybee brain, PDF+ neurons (a circadian-clock output peptide),
0.162 µm/pixel. This is a full-resolution region of a 38 GB whole-brain stack, analysed on
a laptop in minutes.

**Honest scope:** one region, not the whole brain. Bouton count and axon length are real,
standard metrics; the 3D object count is per-region. That framing — a working pipeline on
real data — is stronger than overclaiming a finished census.